## Fake News Detection

In [85]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [86]:
true_df = pd.read_csv('True.csv')
fake_df = pd.read_csv('Fake.csv')

In [87]:
true_df['label'] = 0

In [88]:
fake_df['label'] = 1

In [89]:
true_df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0


In [90]:
fake_df.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [91]:
true_df = true_df[['text','label']]
fake_df = fake_df[['text','label']]

In [92]:
dataset = pd.concat([true_df , fake_df])

In [93]:
dataset.shape

(44898, 2)

### Null values

In [94]:
dataset.isnull().sum() # no null values

text     0
label    0
dtype: int64

### Balanced or Unbalanced dataset

In [95]:
dataset['label'].value_counts()

label
1    23481
0    21417
Name: count, dtype: int64

In [96]:
true_df.shape # true news

(21417, 2)

In [97]:
fake_df.shape # fake news

(23481, 2)

### Shuffle or Resample

In [98]:
dataset = dataset.sample(frac = 1)

In [99]:
dataset.head(20)

,text,label
17623,This story is about more than a massive cover-...,1
1002,WASHINGTON (Reuters) - The U.S. House of Repre...,0
18209,WASHINGTON (Reuters) - Pakistan s foreign mini...,0
8608,WASHINGTON (Reuters) - WikiLeaks released file...,0
5059,Floundering Republican presidential candidate ...,1
17800,MANILA (Reuters) - Philippine President Rodrig...,0
19834,"Wow! If I were Hillary, I d stop sending peopl...",1
2018,The entire Republican Party spent the 2016 cam...,1
5276,WASHINGTON (Reuters) - While President Donald ...,0
15028,LONDON (Reuters) - British ministers rallied a...,0


In [100]:
import sys
print(sys.executable)

/Users/garvsurve/Desktop/MAC/ProjectML/.venv/bin/python


In [101]:
import nltk
print(nltk.__version__)

3.9.2


In [102]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [103]:
lemmatizer = WordNetLemmatizer()

In [104]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/garvsurve/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [105]:
from nltk.corpus import stopwords

stop_words = stopwords.words('english')

In [106]:
stopwords = stopwords.words('english')

In [107]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/garvsurve/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [108]:
def clean_data(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)

    token = text.split()

    token = [
        lemmatizer.lemmatize(word)
        for word in token
        if word not in stop_words
    ]

    clean_news = ' '.join(token)

    return clean_news

In [109]:
from nltk.corpus import stopwords

stop_words = stopwords.words('english')

In [110]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [111]:
dataset['text'] = dataset['text'].apply(lambda x: clean_data(x))

In [112]:
dataset.isnull().sum()

text     0
label    0
dtype: int64

In [113]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [114]:
vectorizer = TfidfVectorizer(max_features = 50000 , lowercase=False , ngram_range=(1,2))

In [115]:
X = dataset.iloc[:35000,0]
y = dataset.iloc[:35000,1]

In [116]:
X.head()

17623    story massive cover elected official sexually ...
1002     washington reuters u house representative help...
18209    washington reuters pakistan foreign minister t...
8608     washington reuters wikileaks released file wed...
5059     floundering republican presidential candidate ...
Name: text, dtype: object

In [117]:
y.head()

17623    1
1002     0
18209    0
8608     0
5059     1
Name: label, dtype: int64

In [118]:
from sklearn.model_selection import train_test_split
train_X , test_X , train_y , test_y = train_test_split(X , y , test_size = 0.2 ,random_state = 0)

In [119]:
from sklearn.model_selection import train_test_split

X = dataset["text"]
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(35918,)
(8980,)


In [120]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

vec_train = vectorizer.fit_transform(X_train)
vec_test = vectorizer.transform(X_test)

print(vec_train.shape)
print(vec_test.shape)

(35918, 5000)
(8980, 5000)


In [121]:
vec_test = vectorizer.transform(test_X).toarray()

In [122]:
print(type(vec_train))
print(vec_train.shape)
print(type(vec_test))
print(vec_test.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(35918, 5000)
<class 'numpy.ndarray'>
(7000, 5000)


In [123]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

vec_train = vectorizer.fit_transform(X_train)
vec_test = vectorizer.transform(X_test)

print("Train:", vec_train.shape)
print("Test:", vec_test.shape)

Train: (35918, 5000)
Test: (8980, 5000)


## Multinomial NB

In [124]:
from sklearn.naive_bayes import MultinomialNB

In [125]:
from sklearn.metrics import accuracy_score,classification_report

In [126]:
clf = MultinomialNB()

In [127]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fresh TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)

vec_train = vectorizer.fit_transform(X_train)
vec_test = vectorizer.transform(X_test)

print("vec_train:", vec_train.shape)
print("vec_test:", vec_test.shape)


vec_train: (35918, 5000)
vec_test: (8980, 5000)


In [128]:
clf.fit(vec_train, y_train)

predictions = clf.predict(vec_test)

print(predictions.shape)

(8980,)


In [129]:
from sklearn.metrics import classification_report

# Train model
clf.fit(vec_train, y_train)

# Predictions
predictions = clf.predict(vec_test)

# Evaluation
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      4284
           1       0.93      0.94      0.94      4696

    accuracy                           0.93      8980
   macro avg       0.93      0.93      0.93      8980
weighted avg       0.93      0.93      0.93      8980



In [130]:
from sklearn.metrics import accuracy_score

predictions_train = clf.predict(vec_train)

train_accuracy = accuracy_score(y_train, predictions_train)

print("Training Accuracy:", train_accuracy)

Training Accuracy: 0.9332646583885517


In [131]:
accuracy_score(y_test, predictions)

0.9324053452115812